In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb_api_key")


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoTokenizer
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoTokenizer
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
import logging
import os
from transformers.integrations import TensorBoardCallback
from transformers import TrainerCallback

import time

start_time = time.time()



train_df = pd.read_csv("/kaggle/input/nlp-assign/train(1).csv")[:-500]
val_df = pd.read_csv("/kaggle/input/nlp-assign/train(1).csv")[-500:]
test_df = pd.read_csv("/kaggle/input/nlp-assign/test(1).csv")


model_name = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["title"], max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Apply preprocessing
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)


dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})



# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("training_log.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Custom callback to log metrics after each evaluation
class LoggingCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            logger.info(f"Evaluation metrics at step {state.global_step}: {metrics}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            logger.info(f"Logs at step {state.global_step}: {logs}")


# Modify training arguments to include TensorBoard logging
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to=["tensorboard"],
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Add callbacks to the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    callbacks=[TensorBoardCallback(), LoggingCallback()]
)

# Log start of training
logger.info("Starting model training")

# Train and log completion
trainer.train()
logger.info("Training completed")

# Save the model
model_path = "./final_model"
trainer.save_model(model_path)
logger.info(f"Model saved to {model_path}")


end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")


In [ ]:
def generate_title_beam(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, num_beams=5, early_stopping=True)  # Beam search with 5 beams
    return tokenizer.decode(output[0], skip_special_tokens=True)


# Generate Titles for Test Set
test_df["generated_title"] = test_df["text"].apply(generate_title_beam)
test_df.to_csv("test_predictions_beam.csv", index=False)

print("Predictions saved to test_predictions_beam.csv")

In [ ]:
import pandas as pd
from rouge_score import rouge_scorer



# Load the test predictions
# df = pd.read_csv("test_predictions.csv")
df = test_df
# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Function to compute ROUGE scores
def compute_rouge(df):
    rouge1, rouge2, rougeL = [], [], []

    for ref, gen in zip(df["title"], df["generated_title"]):
        scores = scorer.score(str(ref), str(gen))  # Convert to string in case of NaN values
        rouge1.append(scores['rouge1'].fmeasure)
        rouge2.append(scores['rouge2'].fmeasure)
        rougeL.append(scores['rougeL'].fmeasure)

    return {
        "ROUGE-1": sum(rouge1) / len(rouge1),
        "ROUGE-2": sum(rouge2) / len(rouge2),
        "ROUGE-L": sum(rougeL) / len(rougeL)
    }

# Compute ROUGE scores
rouge_scores = compute_rouge(df)
print("ROUGE Scores:", rouge_scores)


# C2

In [ ]:
pip install evaluate

In [ ]:
# Function to generate titles using different prompts
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
# from datasets import load_metric
import time

start_time = time.time()




# Load the Flan-T5 models and tokenizers
flan_t5_base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
flan_t5_base_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

flan_t5_large_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
flan_t5_large_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

def generate_with_prompts(model, tokenizer, dataset, prompt):
    predictions = []
    references = dataset["title"].tolist()
    for text in dataset["text"]:
        # Format text with prompt
        input_text = prompt.format(text=text)
        # Tokenize input text
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
        # Generate predictions
        outputs = model.generate(**inputs, max_length=50, num_beams=5, early_stopping=True)
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(prediction)
    return predictions

# Define prompts
prompt_1 = "Summarize the following text into a title: {text}"
prompt_2 = "Generate a headline for this article: {text}"

# Load ROUGE metric
try:
    # For newer versions of datasets
    from datasets import load_metric
    rouge = load_metric("rouge")
except:
    # For newer versions of evaluate
    from evaluate import load
    rouge = load("rouge")

# Generate titles using Flan-T5 base model and both prompts
base_predictions_1 = generate_with_prompts(flan_t5_base_model, flan_t5_base_tokenizer, test_df, prompt_1)
base_predictions_2 = generate_with_prompts(flan_t5_base_model, flan_t5_base_tokenizer, test_df, prompt_2)

# Generate titles using Flan-T5 large model and both prompts
large_predictions_1 = generate_with_prompts(flan_t5_large_model, flan_t5_large_tokenizer, test_df, prompt_1)
large_predictions_2 = generate_with_prompts(flan_t5_large_model, flan_t5_large_tokenizer, test_df, prompt_2)

# Evaluate using ROUGE metric
base_results_1 = rouge.compute(predictions=base_predictions_1, references=test_df["title"].tolist())
base_results_2 = rouge.compute(predictions=base_predictions_2, references=test_df["title"].tolist())
large_results_1 = rouge.compute(predictions=large_predictions_1, references=test_df["title"].tolist())
large_results_2 = rouge.compute(predictions=large_predictions_2, references=test_df["title"].tolist())

# Print results
print("Flan-T5 Base Predictions with Prompt 1 (first 3):", base_predictions_1[:100])
print("Flan-T5 Base ROUGE Scores with Prompt 1:", base_results_1)
print("\nFlan-T5 Base Predictions with Prompt 2 (first 3):", base_predictions_2[:100])
print("Flan-T5 Base ROUGE Scores with Prompt 2:", base_results_2)
print("\nFlan-T5 Large Predictions with Prompt 1 (first 3):", large_predictions_1[:100])
print("Flan-T5 Large ROUGE Scores with Prompt 1:", large_results_1)
print("\nFlan-T5 Large Predictions with Prompt 2 (first 3):", large_predictions_2[:100])
print("Flan-T5 Large ROUGE Scores with Prompt 2:", large_results_2)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")